# Chapitre 1 — Télécharger les données (clé Kaggle)

Ce notebook **extrait le jeu de données RSNA** via l'API Kaggle. C'est le
prérequis aux chapitres 2.5, 3, 4 et 5.

**Avant de lancer :**
1. Avoir accepté les règles de la compétition sur
   <https://www.kaggle.com/competitions/rsna-breast-cancer-detection/rules>
   (sinon Kaggle renvoie une erreur 403).
2. Les identifiants Kaggle doivent être disponibles : `docker-run.sh` monte
   `~/.kaggle` (avec `kaggle.json`) en lecture seule dans le conteneur. Sinon,
   renseigner `KAGGLE_USERNAME` / `KAGGLE_KEY` (voir `.env.example`).

Le jeu complet (images DICOM) fait **~314 Go** → téléchargement optionnel,
désactivé par défaut. On commence par les labels (`train.csv`, ~2 Mo).

In [ ]:
import os
# Vérifie les identifiants AVANT d'importer kaggle (l'import authentifie aussitôt).
os.environ.setdefault('KAGGLE_CONFIG_DIR', os.path.expanduser('~/.kaggle'))
have_json = os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json'))
have_env = bool(os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'))
assert have_json or have_env, (
    'Identifiants Kaggle absents : monte ~/.kaggle (kaggle.json) ou renseigne '
    'KAGGLE_USERNAME / KAGGLE_KEY (voir README).')
import kaggle  # déclenche l'authentification
print('Authentification Kaggle OK')

In [ ]:
import glob, zipfile

COMP = 'rsna-breast-cancer-detection'
DATA_DIR = os.path.expanduser('~/data/rsna')   # volume monté -> persistant
os.makedirs(DATA_DIR, exist_ok=True)

def _unzip_all(folder):
    """Décompresse puis supprime tous les .zip d'un dossier."""
    for z in glob.glob(os.path.join(folder, '*.zip')):
        print('décompression', os.path.basename(z))
        with zipfile.ZipFile(z) as zf:
            zf.extractall(folder)
        os.remove(z)

print('Cible :', DATA_DIR)

In [ ]:
# --- Labels seuls (léger) : de quoi explorer avant de tout télécharger ---
import pandas as pd
train_csv = os.path.join(DATA_DIR, 'train.csv')
if not os.path.exists(train_csv):
    kaggle.api.competition_download_file(COMP, 'train.csv', path=DATA_DIR, quiet=False)
    _unzip_all(DATA_DIR)
df = pd.read_csv(train_csv)
print('train.csv :', df.shape)
print('prévalence cancer :', round(df['cancer'].mean(), 4))
df.head()

In [ ]:
# --- Jeu COMPLET (images DICOM, ~314 Go) : passer FULL=True pour télécharger ---
FULL = False
if FULL and not os.path.isdir(os.path.join(DATA_DIR, 'train_images')):
    kaggle.api.competition_download_files(COMP, path=DATA_DIR, quiet=False)
    _unzip_all(DATA_DIR)
    print('Téléchargement complet terminé.')
else:
    present = os.path.isdir(os.path.join(DATA_DIR, 'train_images'))
    print('train_images déjà présent :' if present else 'FULL=False (téléchargement complet ignoré).',
          present)